# ViTVS Audio Denoising - Colab Runner
Notebook ini digunakan untuk menjalankan training dan testing secara langsung di Google Colab dengan melakukan *clone* terhadap repositori utama.


### 1. Persiapan Lingkungan (Mount Drive & Clone Repo)


In [1]:
# Mount Google Drive (Opsional jika dataset/checkpoint ada di GDrive)
from google.colab import drive
drive.mount('/content/drive')

# Contoh jika repo sudah di-push ke GitHub, ganti URL di bawah:
!git clone https://github.com/user312982/bird-denoising-new.git
%cd bird-denoising-new

# Install dependencies dari requirements.txt
!pip install -q -r requirements.txt


Mounted at /content/drive
Cloning into 'bird-denoising-new'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 27 (delta 9), reused 27 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 11.80 KiB | 11.80 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/bird-denoising-new
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.5 MB/s eta 0:00:00


### 2. Persiapan Dataset (Opsional)
Jika Anda menaruh dataset di Google Drive, Anda bisa membuat symbolic link agar path-nya sesuai, atau Anda bisa langsung mengoper path tersebut lewat parameter.


In [ ]:
# 1. Membuat struktur direktori lokal untuk menampung data
!mkdir -p data/train/noisy data/train/clean
!mkdir -p data/valid/noisy data/valid/clean
!mkdir -p data/test/noisy data/test/clean
!mkdir -p checkpoints

# --- 2. PROSES TRAIN DATASET ---
print("Mengekstrak dan menyiapkan Train Dataset (3000 file)...")
!unzip -q /content/drive/MyDrive/dataset/Train_subset_3000.zip -d /content/temp_train

# Pindahkan file dengan sangat cepat menggunakan xargs (Raw -> noisy, Denoised -> clean)
!find /content/temp_train -type f -path "*/Raw_audios/*" -print0 | xargs -0 mv -t data/train/noisy/
!find /content/temp_train -type f -path "*/Denoised_audios/*" -print0 | xargs -0 mv -t data/train/clean/
!rm -rf /content/temp_train  # Hapus sampah ekstrak agar Colab tidak penuh

# --- 3. PROSES VALID DATASET ---
print("Mengekstrak dan menyiapkan Valid Dataset (3000 file)...")
!unzip -q /content/drive/MyDrive/dataset/Valid_subset_3000.zip -d /content/temp_valid

!find /content/temp_valid -type f -path "*/Raw_audios/*" -print0 | xargs -0 mv -t data/valid/noisy/
!find /content/temp_valid -type f -path "*/Denoised_audios/*" -print0 | xargs -0 mv -t data/valid/clean/
!rm -rf /content/temp_valid

print("Persiapan Dataset 3000 File Selesai! Proses ini seharusnya memakan waktu kurang dari 1 Menit.")


### 3. Training Model
Jalankan `train.py`. Model akan otomatis mencari checkpoint terbaru di dalam folder `checkpoints/` untuk *resume training* jika Colab terputus.


In [19]:
# Jalankan proses training beserta direktori validasi
# Anda dapat menggunakan default path, atau spesifikasikan direktori dari GDrive
!python train.py \
    --epochs 50 \
    --batch_size 4 \
    --lr 0.0001 \
    --noisy_dir "data/train/clean" \
    --clean_dir "data/train/noisy" \
    --val_noisy_dir "data/valid/clean" \
    --val_clean_dir "data/valid/noisy" \
    --checkpoint_dir "/content/drive/MyDrive/checkpoints/vitvs_denoising"



Streaming output truncated to the last 5000 lines.
                                                               train_loss_epoch:
Epoch 48/49 ━╸━━━━━━━━━━━━━━ 15/150 0:00:04 • 0:00:45 3.04it/s v_num: 5.000     
                                                               train_loss_step: 
                                                               0.143 val_loss:  
                                                               2.594            
                                                               train_loss_epoch:
Epoch 48/49 ━╸━━━━━━━━━━━━━━ 15/150 0:00:05 • 0:00:45 3.04it/s v_num: 5.000     
                                                               train_loss_step: 
                                                               0.185 val_loss:  
                                                               2.594            
                                                               train_loss_epoch:
Epoch 48/49 ━╸━━━━━━━━━━━━━━ 16/150 0:00:05 • 0:00:45 3.03

### 4. Evaluasi / Testing (Denoising)


In [18]:
# Pilih checkpoint terbaik atau terakhir dari training
ckpt_path = "/content/drive/MyDrive/checkpoints/vitvs_denoising/last.ckpt"

# Jalankan skrip test
!python test.py \
    --ckpt "$ckpt_path" \
    --input "data/train/clean/XC100018_left.wav" \
    --output "hasil_denoising.wav"

# Mainkan audio hasil denoising langsung di Colab
import IPython.display as ipd
ipd.Audio('hasil_denoising.wav')


Loading model weights from /content/drive/MyDrive/checkpoints/vitvs_denoising/last.ckpt...
Processing audio: data/train/clean/XC100018_left.wav
Success! Denoised audio saved to: hasil_denoising.wav


In [17]:
!rm -rf src/__pycache__
!git pull

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 4), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.34 KiB | 599.00 KiB/s, done.
From https://github.com/user312982/bird-denoising-new
   2001328..919cc64  main       -> origin/main
Updating 2001328..919cc64
Fast-forward
 .gitignore |  1 +
 train.py   | 18 ++++++++++++++----
 2 files changed, 15 insertions(+), 4 deletions(-)
 create mode 100644 .gitignore
